# Notebook 3 — Progressive Reveal

---

## How MRI Scanners Actually Acquire Data

MRI scanners don't capture a full image in one shot. They fill in k-space **line by line**, starting from the center and working outward. This takes time — anywhere from seconds to minutes depending on the scan.

The order matters:

```
First acquired → center of k-space → coarse shape, bulk anatomy
Last acquired  → edges of k-space  → fine detail, sharp edges
```

This means if a patient moves during the scan, the edges of k-space are corrupted — but the center (acquired first) may still be clean. That's why motion artifacts often look like blurring or ghosting rather than a total loss of image.

---

## The Idea: Watch the Image Build

If we reconstruct the image at each step of acquisition — using only the k-space data collected so far — we can watch the image build from nothing to full resolution.

This is **progressive reveal**: reconstructing from increasingly large portions of k-space.

```
12% of k-space → blurry blob
25% of k-space → recognizable anatomy
50% of k-space → most detail visible
100% of k-space → full resolution
```

---

## Why This Is Useful

**For engineering:** This is exactly how **compressed sensing** works — modern fast MRI scanners only acquire 25–30% of k-space and use algorithms to fill in the rest. Progressive reveal shows you the baseline: what does that 25% actually look like without any reconstruction tricks?

**For patients:** Showing a scan building from a blur to a sharp image is one of the most intuitive ways to explain MRI to someone who has never seen one. Instead of "this is your knee" (static, confusing), you get "this is your knee assembling itself" (dynamic, memorable).

**For diagnosis:** How much of k-space do you actually need before the anatomy is legible? This notebook gives you a visual answer.

---

## Key Term: DC Component

The very center point of k-space — the single pixel at position (0,0) — is called the **DC component** (from electrical engineering: Direct Current). It holds the average brightness of the entire image. It's the most powerful single point in k-space and the first thing acquired.

When you see a bright dot in the center of a k-space visualization, that's the DC component.

---

## Step 1 — Import Libraries

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from kode.io import load_kspace
from kode.progressive_reveal import progressive_frames

---

## Step 2 — Load K-Space and Generate Frames

We load one slice and generate 8 reconstructions — each using a larger low-frequency region than the last.

- Frame 1 uses the central 12.5% of k-space
- Frame 2 uses the central 25%
- ...
- Frame 8 uses 100% (full reconstruction)

`progressive_frames()` applies a circular low-frequency mask at each step and returns a list of 2D images.

In [ ]:
H5_FILE = '../data/singlecoil_test/file1000022.h5'
kspace = load_kspace(H5_FILE, slice_idx=15)
print(f'K-space shape: {kspace.shape}  →  (coils, height, width)')

frames = progressive_frames(kspace, steps=8)
print(f'Generated {len(frames)} frames — one per acquisition stage')

---

## Step 3 — Display the Reveal Sequence

Each panel below shows the image at a different stage of k-space coverage. Watch how anatomy emerges — first as a vague shape, then gaining edges and detail as more frequency content is added.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
percentages = [f'{int((i+1)/8*100)}% of k-space' for i in range(8)]

for ax, frame, label in zip(axes.flat, frames, percentages):
    ax.imshow(frame, cmap='gray')
    ax.set_title(label, fontsize=10)
    ax.axis('off')

plt.suptitle('Progressive Reveal — From DC Component to Full Resolution', fontsize=13)
plt.tight_layout()
plt.savefig('../results/progressive_reveal.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to results/progressive_reveal.png')

---

## Step 4 — Export as an Animated GIF

A static grid is informative. An animation is memorable. This cell saves the reveal sequence as a looping GIF — ready to embed in a README or show in a presentation.

The GIF loops from blurry → sharp → blurry, making the frequency contribution of each k-space region immediately visible.

In [ ]:
try:
    from PIL import Image

    gif_frames = []
    for frame in frames:
        normalized = ((frame / frame.max()) * 255).astype('uint8')
        gif_frames.append(Image.fromarray(normalized))

    gif_frames[0].save(
        '../results/progressive_reveal.gif',
        save_all=True,
        append_images=gif_frames[1:],
        duration=300,
        loop=0,
    )
    print('GIF saved to results/progressive_reveal.gif')
    print('Embed this in the README — it demonstrates the concept instantly.')
except ImportError:
    print('Pillow not installed — run: pip3 install Pillow')

---

## What This Means

The progressive reveal demonstrates something fundamental about MRI data: **information is not uniformly distributed across k-space**.

- The central 10% of k-space contains most of the diagnostic information
- The outer 50% contributes mostly fine texture and edge sharpness
- The very outer edges contribute noise as much as signal

This is why compressed sensing can reconstruct a full image from 25% of k-space data. It's also why motion artifacts are so damaging — they corrupt the edges, which are acquired last and can't be re-acquired mid-scan without starting over.